# Test a Cellpose model

**What it does.** Score a Cellpose model against ground-truth masks.

**When to use it.** Before trusting a fine-tuned model on a whole plate. Compare candidates on the same held-out images.

**What you get.** Per-image segmentation metrics and a summary table.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.test_cellpose_model`

```
test_cellpose_model(settings)
```

Evaluate a Cellpose model on a labelled test set and report per-image metrics.

In [ ]:
from spacr.submodules import test_cellpose_model

## 3. Settings and API reference

Read the descriptions here, then edit only the values in the next cell. Defaults and descriptions are generated from the installed spaCR version, so the notebook stays aligned with the API.

### [`spacr.submodules.test_cellpose_model`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.test_cellpose_model)

- **`CP_probability`** — (int) - Cellpose cellprob_threshold used by the standalone apply/test-model submodules, where it carries this name instead of the per-object &lt;object&gt;_CP_prob used by the Mask module. Only pixels whose predicted cell probability exceeds it join a mask, so raising it shrinks outlines and drops faint objects while lowering it grows them and recovers dim ones. Default 0.
- **`FT`** — (int) - Cellpose flow_threshold for the standalone apply/test-model submodules, the counterpart of the Mask module's per-object &lt;object&gt;_FT. Masks whose recomputed flows disagree with the network's prediction by more than this are discarded, so a low value strips ragged or implausible objects and also loses real ones. Default 100, which effectively accepts every candidate.
- **`batch_size`** — (int) - How many images are held and processed together in one pass: field stacks during normalization and Cellpose segmentation, crops per step during classifier training and activation maps. Raising it speeds runs up but increases RAM/VRAM roughly linearly; lower it on out-of-memory errors. Defaults: 50 for mask generation, 64 for training.
- **`model_path`** — (str) - Path to a trained spaCR classifier saved as a whole PyTorch object (loaded with torch.load(weights_only=False), not a state_dict). Used when applying a model to a dataset tar and when generating activation maps. deep_spacr overwrites it with the freshly trained model whenever train is True, so set it only to score with an existing model. Default ''.
- **`normalize`** — (bool) - Percentile-normalize each image channel (2nd to 98th percentile, clipped to 0-1) before display or model input; in the activation-map tool this rescales the image the CAM/saliency heatmap is drawn over. Turn it on when raw channels are too dim to read under the overlay. Affects display and input scaling only, never stored pixels. Default True.
- **`percentiles`** — (list) - Two percentiles [low, high] used to rescale each channel of each image to 0-1 before segmentation, e.g. [2, 98]. Narrowing the window boosts contrast on dim objects but clips bright ones. Set None to derive them automatically: low fixed at 2, high the first of 98/99/99.9/99.99/99.999 exceeding background * Signal_to_noise. Default None in the Cellpose steps.
- **`save`** — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.
- **`src`** — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`target_size`** — (int) - Edge length in pixels that the training images and masks are resized to before Cellpose fine-tuning, applied to both axes so the input becomes square. Larger keeps fine boundary detail and costs VRAM and time roughly quadratically; smaller trains faster and blurs exactly the outlines the model is being taught. Default 1000.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    'CP_probability': 0,
    'FT': 100,
    'batch_size': 50,
    'model_path': 'path',
    'normalize': True,
    'percentiles': (2, 98),
    'save': True,
    'src': 'path',
    'target_size': 1000,
}

In [ ]:
test_cellpose_model(settings)

## Where the output went

Per-image segmentation metrics and a summary table.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.